# 교차 검증과 그리드 서치

<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/rickiepark/hg-mldl/blob/master/5-2.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />구글 코랩에서 실행하기</a>
  </td>
</table>

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from scipy.stats import uniform, randint

## 검증 세트

In [2]:
# wine = pd.read_csv('https://bit.ly/wine_csv_data')
wine = pd.read_csv('./wine.csv')

data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()
print(data.shape, target.shape)

(6497, 3) (6497,)


In [3]:
train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42
)
print(train_input.shape, test_input.shape)
print(test_target.shape, train_target.shape)

sub_input, val_input, sub_target, val_target = train_test_split(
    train_input, train_target, test_size=0.2, random_state=42
)
print(sub_input.shape, val_input.shape)
print(val_target.shape, sub_target.shape)

(5197, 3) (1300, 3)
(1300,) (5197,)
(4157, 3) (1040, 3)
(1040,) (4157,)


In [4]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)

print(dt.score(sub_input, sub_target))
print(dt.score(val_input, val_target))

0.9971133028626413
0.864423076923077


## 교차 검증

In [5]:
scores = cross_validate(dt, train_input, train_target)
scores

{'fit_time': array([0.00532794, 0.00498366, 0.00588059, 0.00531721, 0.00488043]),
 'score_time': array([0.00119591, 0.00111818, 0.00105906, 0.0013628 , 0.00125837]),
 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}

In [8]:
print(scores['test_score'].mean())
print(np.mean(scores['test_score']))

0.855300214703487
0.855300214703487


In [7]:
scores = cross_validate(dt, train_input, train_target, cv=StratifiedKFold())
print(np.mean(scores['test_score']))

0.855300214703487


In [9]:
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_validate(dt, train_input, train_target, cv=splitter)
print(np.mean(scores['test_score']))

0.8574181117533719


## 하이퍼파라미터 튜닝

In [10]:
params = {
    'min_impurity_decrease': [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]
}
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)
gs.fit(train_input, train_target)
print(gs.best_score_)

dt = gs.best_estimator_
print(dt.score(train_input, train_target))
print(gs.best_params_)
print(gs.cv_results_['mean_test_score'])

best_index = np.argmax(gs.cv_results_['mean_test_score'])
print(gs.cv_results_['params'][best_index])

0.8681929740134745
0.9615162593804117
{'min_impurity_decrease': 0.0001}
[0.86819297 0.86453617 0.86492226 0.86780891 0.86761605]
{'min_impurity_decrease': 0.0001}


In [11]:
params = {
    'min_impurity_decrease': np.arange(0.0001, 0.001, 0.0001),
    'max_depth': range(5, 20, 1),
    'min_samples_split': range(2, 100, 10),
}
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)
gs.fit(train_input, train_target)
print(gs.best_score_)
print(gs.best_params_)
print(np.max(gs.cv_results_['mean_test_score']))

0.8683865773302731
{'max_depth': 14, 'min_impurity_decrease': np.float64(0.0004), 'min_samples_split': 12}
0.8683865773302731


### 랜덤 서치

In [14]:
rgen = randint(0, 10)
rgen.rvs(10)

array([0, 9, 1, 2, 8, 3, 5, 0, 0, 8])

In [17]:
np.unique(rgen.rvs(1000), return_counts=True)

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([ 83, 113,  94,  96,  92, 108, 107, 106, 108,  93]))

In [21]:
ugen = uniform(0, 1)
ugen.rvs(10)

array([0.52048939, 0.16980227, 0.70219765, 0.38529166, 0.15269017,
       0.92191399, 0.57316358, 0.49678197, 0.1452697 , 0.06347519])

In [22]:
params = {
    'min_impurity_decrease': uniform(0.0001, 0.001),
    'max_depth': randint(20, 50),
    'min_samples_split': randint(2, 25),
    'min_samples_leaf': randint(1, 25),
}
gs = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42),
    params, n_iter=100, n_jobs=-1, random_state=42
)
gs.fit(train_input, train_target)
print(gs.best_score_)
print(gs.best_params_)
print(np.max(gs.cv_results_['mean_test_score']))

dt = gs.best_estimator_
print(dt.score(test_input, test_target))

0.8695428296438884
{'max_depth': 39, 'min_impurity_decrease': np.float64(0.00034102546602601173), 'min_samples_leaf': 7, 'min_samples_split': 13}
0.8695428296438884
0.86


## 확인문제

In [23]:
gs = RandomizedSearchCV(
    DecisionTreeClassifier(splitter='random', random_state=42),
    params, n_iter=100, n_jobs=-1, random_state=42
)
gs.fit(train_input, train_target)
print(gs.best_score_)
print(gs.best_params_)
print(np.max(gs.cv_results_['mean_test_score']))

dt = gs.best_estimator_
print(dt.score(test_input, test_target))

0.8458726956392981
{'max_depth': 43, 'min_impurity_decrease': np.float64(0.00011407982271508446), 'min_samples_leaf': 19, 'min_samples_split': 18}
0.8458726956392981
0.786923076923077
